In [ ]:
from datasets import load_from_disk
import random
import outlines


In [ ]:
outlines.caching.disable_cache()
init_rating = 1000

In [ ]:
class Player():
    def __init__(self, rating):
        self.rating = rating
    def update_rating(self, expected_value, true_score):
        pass

def calculate_expected_value(player_a, player_b):
    denom = 1 + 10 ** ((player_b.rating - player_a.rating) / 400)
    return 1 / denom



In [ ]:
import os

base = "definition_task_outputs/"

def flan_name(type, id):
    return f"flan_{type}_new_token_definitions_{id}"


In [ ]:
baseline_lr_0003_fnames = ["baseline_generations_lr_0.0003_bea6ecf2-2fab-443d-a096-98b12357b84e",
"baseline_generations_lr_0.0003_ef46996f-7519-48ee-adaf-b97c4f7c72ca",
"baseline_generations_lr_0.0003_f932d5ed-bbc5-4180-8d64-d2986455f62f"]

baseline_lr_001_fnames = ["baseline_generations_lr_0.001_5d4a9510-503b-4501-8a05-c6e1a3672fff",
"baseline_generations_lr_0.001_6b71232d-ec9c-4906-8ded-58ae9dfa2327",
"baseline_generations_lr_0.001_c1fe724e-7e54-4666-9cc0-9fa522751924"]

baseline_no_lr_fnames = ["baseline_generations_no_lr_7b842393-a953-444c-9de0-f45520f643fe",
"baseline_generations_no_lr_86c35d06-fe52-48f9-b7cd-f9c92eb00097",
"baseline_generations_no_lr_14d778d6-ad36-470a-a1d8-e39c3464e0fa"]

hice_fnames = ["hice_generations_1462255a-5223-4f70-9e66-daa827536008",
"hice_generations_9ba894ef-e436-4395-a151-b093834a7738",
"hice_generations_ac9c443a-3b9d-44c1-871a-a8d01a9850f8"]

additive_fnames = ["additive_generations_12e6e655-4621-41d0-a54a-ce9a38be5672",
"additive_generations_3681250e-28ff-49d4-9d2c-4a77ddb59e51",
"additive_generations_7233e864-cd8b-450f-bb4d-c2c1174368e6"]

emb_gen_fnames = ["emb_gen_generations_masked_new_token_new_data_new_model_90318e95-91c0-479d-8ab9-a50a022b519a",
"emb_gen_generations_masked_new_token_new_data_new_model_a7a9fae8-1153-4b96-92ce-29fa6ab23602",
"emb_gen_generations_masked_new_token_new_data_new_model_a9d91273-5e88-468d-8a24-d9e3778f00e2"]

flan_base_fnames = [flan_name("base", i) for i in range(3)]
flan_large_fnames = [flan_name("large", i) for i in range(3)]
flan_xl_fnames = [flan_name("xl", i) for i in range(3)]

print(flan_base_fnames)
print(flan_large_fnames)
print(flan_xl_fnames)
base = "definition_task_outputs/" 

In [ ]:
def prepare_baseline(base, name, with_prompt):
    data = load_from_disk(base + name)
    if with_prompt:
        return data.filter(lambda ex: "Given the following" in ex['prompt'])
    else:
        return data.filter(lambda ex: "Given the following" not in ex['prompt'])

def prepare_flan_baseline(base, name):
    return load_from_disk(base + name)

In [ ]:
from dataclasses import dataclass

@dataclass
class Battle():
    model_a: str
    model_b: str
    winner: str
    k_shot: int
    model_a_generation: str
    model_b_generation: str
    word: str
    true_definition: str

In [ ]:
model = outlines.models.openai("gpt-3.5-turbo", api_key="")


In [ ]:
with_prompt = False
if with_prompt:
    baseline_model_names = ["lr_0003_step_1", "lr_0003_step_2", "hice", "additive"]
    baseline_fnames = [hice_fnames, additive_fnames, baseline_no_lr_fnames]
else:
    baseline_model_names = ["lr_0003_step_1", "lr_0003_step_2", "hice", "additive"]
    baseline_fnames = [hice_fnames, additive_fnames]
emb_gen = [prepare_baseline(base, n, with_prompt) for n in emb_gen_fnames]
baselines = [[prepare_baseline(base, n, with_prompt) for n in fn] for fn in baseline_fnames]

tt_baselines = [prepare_baseline(base, n, with_prompt) for n in baseline_lr_0003_fnames]
step_1 = [t.filter(lambda ex: ex['step'] == 1) for t in tt_baselines]
step_2 = [t.filter(lambda ex: ex['step'] == 2) for t in tt_baselines]

flan_baselines = [[prepare_flan_baseline(base, n) for n in names_subset] for names_subset in flan_baseline_fnames]
baseline_model_names = baseline_model_names + ["flan_base", "flan_large", "flan_xl"]

baselines = [step_1, step_2] + baselines
baselines = baselines + flan_baselines
prompting_baseline = [prepare_baseline(base, n, with_prompt=True) for n in baseline_no_lr_fnames]
baselines = baselines + [prompting_baseline]
baseline_model_names = baseline_model_names + ["baseline_prompting"]

print(baseline_model_names)

In [ ]:
from tqdm import tqdm


all_battles = []
def_task = load_from_disk("def_task_954")
prompt_format = "Which of the following is a better definition for the word '{}'? A) {}, B) {}, or C) {}."

for trial in range(3):
    battles = []
    for example_index, example in enumerate(def_task):
        order = ['baseline', 'emb_gen', 'tie']
        challenger_idx = random.randint(0, len(baseline_model_names)-1)
        challenger_name = baseline_model_names[challenger_idx]
        challenger_dataset = baselines[challenger_idx][trial]

        emb_gen_dataset = emb_gen[trial]
        k = random.randint(0, 2) + 1
        
        if "flan" not in challenger_name:
            one_step_ex = challenger_dataset.filter(lambda ex: ex['num_examples'] == k).filter(lambda ex: ex['word'] == example['word'])
        else:
            one_step_ex = challenger_dataset.filter(lambda ex: ex['word'] == example['word'])

        emb_gen_ex = emb_gen_dataset.filter(lambda ex: len(ex['examples']) == k).filter(lambda ex: ex['word'] == example['word']) 
        
        challenger_def = one_step_ex[0]['generated definition'].split(".")[0]
        
        emb_gen_def = emb_gen_ex[0]['generated definition'].split(".")[0]
        
        random.shuffle(order)
                
        values = []
        for i in order:
            if i == "baseline":
                values.append(challenger_def)
            elif i == "emb_gen":
                values.append(emb_gen_def)
            elif i == "tie":
                values.append("Tie")
                
        choice_prompt = prompt_format.format(example['word'], values[0], values[1], values[2])
        result = model.generate_choice(choice_prompt, ['A', 'B', 'C'])
        res_idx = ['A', 'B', 'C'].index(result)
        winner = order[res_idx]
        if winner == "baseline":
            win_name = challenger_name
        elif winner == "emb_gen":
            win_name = "emb_gen"
        elif winner == "tie":
            win_name = "tie"
        
        outcome = Battle(model_a = "emb_gen", 
                         model_b = challenger_name, 
                         winner=win_name, 
                         k_shot=k,
                         model_a_generation = emb_gen_def,
                        model_b_generation = challenger_def,
                         word= example['word'],
                        true_definition= example['definition'])
        battles.append(outcome)
        if example_index % 300 == 0:
            print(f"Challenger name {challenger_name}")
            print(f"Order {order}")
            print(f"Prompt {choice_prompt}")
            print(f"Outcome {outcome}")

#         break
#     break
    all_battles.append(battles)
        

In [ ]:
from collections import defaultdict

def compute_elo(battles, K=4, SCALE=400, BASE=10, INIT_RATING=1000):
    rating = defaultdict(lambda: INIT_RATING)

#     for rd, model_a, model_b, winner in battles[['model_a', 'model_b', 'winner']].itertuples():
    for outcome in battles:
        model_a = outcome.model_a
        model_b = outcome.model_b
        winner = outcome.winner
        ra = rating[model_a]
        rb = rating[model_b]
        ea = 1 / (1 + BASE ** ((rb - ra) / SCALE))
        eb = 1 / (1 + BASE ** ((ra - rb) / SCALE))
        if winner == "emb_gen":
            sa = 1
        elif winner == model_b:
            sa = 0
        elif winner == "tie":
            sa = 0.5
        else:
            raise Exception(f"unexpected vote {winner}")
        rating[model_a] += K * (sa - ea)
        rating[model_b] += K * (1 - sa - eb)

    return rating